In [1]:
import torch
import torch.nn as nn
from torch.ao.quantization.quantize_fx import prepare_qat_fx, convert_fx
from torch.ao.quantization import get_default_qat_qconfig_mapping

# 1. Define foundational high-precision target architecture
class VolatileBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(32)
        self.relu = nn.ReLU()
        self.fc = nn.Linear(32 * 32 * 32, 2)
        
    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

# Initialize pre-trained structural baseline instance
model_fp32 = VolatileBlock().train()

# 2. Configure QAT Framework Backend Mappings
# This maps out the specific observers and fake quantization modules for the x86 engine
qat_qconfig_mapping = get_default_qat_qconfig_mapping("x86")

# 3. Transform the Graph into a QAT Structure
# prepare_qat_fx fuses Conv+BN+ReLU blocks and injects FakeQuantize nodes automatically
example_inputs = torch.randn(2, 3, 32, 32)
prepared_qat_model = prepare_qat_fx(model_fp32, qat_qconfig_mapping, example_inputs)

print("--- QAT Graph Transformation Complete ---")
print("FakeQuantize layers and Straight-Through Estimators successfully injected.\n")

# 4. Simulated Quantization-Aware Fine-Tuning Loop
optimizer = torch.optim.AdamW(prepared_qat_model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

# Mock batch matching target distributions
mock_X = torch.randn(2, 3, 32, 32)
mock_y = torch.tensor([0, 1])

# Run active training steps
optimizer.zero_grad()
# Forward pass: runs through fake quantization to introduce noise
predictions = prepared_qat_model(mock_X)
loss = criterion(predictions, mock_y)
# Backward pass: uses STE to bypass zero-gradient rounding constraints
loss.backward()
optimizer.step()

print(f"QAT Optimization Step Executed. Computed Loss Value: {loss.item():.4f}\n")

# 5. Lock Parameters and Convert to Quantized INT8 Architecture
prepared_qat_model.eval()
# convert_fx removes the fake quantization math, replacing it with true production INT8 layers
quantized_production_model = convert_fx(prepared_qat_model)

print("--- Production INT8 Graph Compilation Complete ---")
print(quantized_production_model)

/tmp/ipykernel_1133/3643900685.py:33: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  prepared_qat_model = prepare_qat_fx(model_fp32, qat_qconfig_mapping, example_inputs)


--- QAT Graph Transformation Complete ---
FakeQuantize layers and Straight-Through Estimators successfully injected.

QAT Optimization Step Executed. Computed Loss Value: 0.7910

--- Production INT8 Graph Compilation Complete ---
GraphModule(
  (conv): QuantizedConvReLU2d(3, 32, kernel_size=(3, 3), stride=(1, 1), scale=0.03340879827737808, zero_point=0, padding=(1, 1))
  (fc): QuantizedLinear(in_features=32768, out_features=2, scale=0.002737714909017086, zero_point=3, qscheme=torch.per_channel_affine)
)



def forward(self, x):
    conv_input_scale_0 = self.conv_input_scale_0
    conv_input_zero_point_0 = self.conv_input_zero_point_0
    quantize_per_tensor = torch.quantize_per_tensor(x, conv_input_scale_0, conv_input_zero_point_0, torch.quint8);  x = conv_input_scale_0 = conv_input_zero_point_0 = None
    conv = self.conv(quantize_per_tensor);  quantize_per_tensor = None
    flatten = torch.flatten(conv, 1);  conv = None
    fc = self.fc(flatten);  flatten = None
    dequantize_3 = fc

/usr/local/lib/python3.11/site-packages/torch/ao/quantization/observer.py:534: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  super().__init__(
/tmp/ipykernel_1133/3643900685.py:60: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_production_m